# Exercise 4 · Build a dataset for a new domain

**Part 05 · Fine-tuning** · 10 minutes

You will build training data for **airline customer messages** (or your own domain from the opening
poll) the same way the workshop model's data was built:

* option descriptions, and at least **4 templates per label**, including **one hard case** per label
  (misleading keywords, sarcasm, an instruction inside the message, ...)
* labels set **by construction**: a template written for `refunds` only produces refund messages
* the **last template of each label is validation-only**, and nothing may be a near-duplicate of a test case

Templates can be plain strings with `{slots}` or Python lambdas.

In [ ]:
# Setup: find the workshop folder and import the helpers (works locally and in Colab)
import os, sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/decision-models-workshop")]
             if (p / "dmw").is_dir()), None)
assert ROOT, "Open this notebook from the workshop folder. In Colab, run 00_setup.ipynb first."
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from dmw import config
config.load_env()

In [ ]:
import json
import pandas as pd
from dmw import datagen, checks

test_cases = datagen.load_test("data/airline_test.json")
TEST_TEXTS = [c["text"] for c in test_cases]
print(len(test_cases), "test messages, e.g.:", TEST_TEXTS[:3])

## Your domain

In [ ]:
DOMAIN = {
    "prefix": "Airline customer message: ",
    "question": "Which team should handle this message?",
    "choices": {
        "baggage": "lost, delayed or damaged bags, baggage allowance and fees",
        "booking changes": "changing dates, flights, names or seats on an existing booking",
        "loyalty": "frequent-flyer points, status tiers and member benefits",
        "refunds": "money back for cancelled flights, unused tickets or extra charges",
    },
    "slots": {
        "city": ["Madrid", "Tallinn", "Cairo", "Toronto", "Riga", "Athens", "Dublin"],
        "day": ["Monday", "Thursday", "Saturday", "next week", "the 14th"],
    },
    "templates": {
        "baggage": [
            "My bag never arrived in {city}.",          # example: keep or replace
            # TODO: at least 3 more, one of them a hard case
        ],
        "booking changes": [
            # TODO: at least 4
        ],
        "loyalty": [
            # TODO: at least 4
        ],
        "refunds": [
            # TODO: at least 4
        ],
    },
    "valid_templates_per_label": 1,
}

## Render, split, check

In [ ]:
train_s, valid_s = datagen.scenarios_from_domain(DOMAIN, per_template=8)
rows = datagen.to_rows(train_s) + datagen.to_rows(valid_s, seed=7)
print(len(train_s), "train scenarios |", len(valid_s), "validation scenarios |", len(rows), "rows")
checks.validate_dataset(DOMAIN, train_s, valid_s, TEST_TEXTS, rows)

In [ ]:
print(json.dumps(datagen.manifest(train_s, valid_s, datagen.to_rows(train_s), datagen.to_rows(valid_s, seed=7), []), indent=1))
pd.DataFrame(train_s)[["expected", "template", "text"]].sample(min(10, len(train_s)), random_state=0) if train_s else "no scenarios yet"

## Optional after the session: retrain with your domain and push v2

Combine your rows with the workshop data, rerun notebook 07 with `train_rows + your_rows`, and push to a
new repo name (for example `...-decision-router-v2`). Evaluate on `data/airline_test.json` as well as
`data/fresh_test.json`, so you can see whether the new domain cost accuracy on the old one.

In [ ]:
# Save your scenarios so notebook 07 can pick them up: datagen.to_rows(json.load(...))
json.dump({"domain": {k: v for k, v in DOMAIN.items() if k != "templates"}, "train": train_s, "valid": valid_s},
          open(config.OUTPUTS / "my_domain_scenarios.json", "w"), indent=1, default=str)
print("saved outputs/my_domain_scenarios.json")